# Hybrid CNN+ViT: 10 Research Ideas - Complete Training Pipeline

**Goal**: Train and evaluate 10 novel CNN+ViT hybrid architecture variants on APTOS 2019 diabetic retinopathy dataset.

**Features**:
- ✅ Checkpoint system: Resume interrupted training
- ✅ All metrics: Accuracy, F1-score, QWK, per-class scores
- ✅ Auto-visualization: Loss curves, confusion matrices
- ✅ Best model tracking for each variant

**Dataset Split**: Train 80% | Validation 10% | Test 10%

In [1]:
# 1. Import Required Libraries
import os
import sys
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from typing import Dict, Tuple, List
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Setup
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"PyTorch Version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

✓ Libraries imported successfully
PyTorch Version: 2.9.1+cu126
GPU Available: True
GPU Device: NVIDIA RTX A5000


In [2]:
# 2. Setup Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
results_dir = Path("results/hybrid_cnn_vit")
results_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Results Directory: {results_dir}")
print(f"🔧 Device: {device}")

# Verify model files exist
required_files = [
    "models/hybrid_cnn_vit_base.py",
    "model_configs.py",
    "dataset_loader.py",
    "metrics.py"
]

print("\n📋 Checking required files:")
for file in required_files:
    exists = Path(file).exists()
    status = "✓" if exists else "✗"
    print(f"  {status} {file}")


📁 Results Directory: results\hybrid_cnn_vit
🔧 Device: cuda

📋 Checking required files:
  ✓ models/hybrid_cnn_vit_base.py
  ✓ model_configs.py
  ✓ dataset_loader.py
  ✓ metrics.py


In [ ]:
# 3. Import Local Modules
import importlib
import sys

# Remove cached modules to force reload
if 'models.hybrid_cnn_vit_base' in sys.modules:
    del sys.modules['models.hybrid_cnn_vit_base']
if 'model_configs' in sys.modules:
    del sys.modules['model_configs']

from models.hybrid_cnn_vit_base import HybridCNNViTBase
from model_configs import get_model_config, list_all_models
from dataset_loader import APTOS2019DatasetLoader, get_data_loaders
from metrics import compute_metrics
from sklearn.metrics import confusion_matrix, classification_report

print("✓ Local modules imported successfully (fresh reload)")

# Display all 10 models
models_list = list_all_models()
print(f"\n🎯 Available Models ({len(models_list)}):")
for i, model_name in enumerate(models_list, 1):
    config = get_model_config(model_name)
    print(f"  {i}. {config['name'][:45]:<45s} - {config['description'][:40]}")

✓ Local modules imported successfully

🎯 Available Models (10):
  1. Confidence-Gated Local-Global Fusion          - Add confidence-gated fusion block that l
  2. Lesion-Scale Spatial-Channel Co-Attention Pyr - Multi-scale co-attention combining chann
  3. Uncertainty-Driven Token Refinement Transform - Iteratively prune/reweight uncertain tok
  4. Ordinal-Distribution Aware QWK Optimization   - Hybrid objective with ordinal dist., QWK
  5. Dual-Stream Cross-Attention with Lesion Proto - Forces both branches to align through cr
  6. Topology-Aware Retinal Graph Transformer (Str - Graph attention network over lesion cand
  7. Frequency-Spatial Dual Domain Hybrid Encoder  - Parallel frequency branch with cross-dom
  8. Mixture-of-Experts Severity Router (Complex)  - Expert sub-networks specialized for low/
  9. Causal Counterfactual Lesion Consistency      - Lesion-focused counterfactual consistenc
  10. Tri-Level Uncertainty-Calibrated Self-Distill - Multi-level distillation with uncert

In [9]:
# 4. Load Dataset
print("\n📊 Loading APTOS2019 Dataset...")

batch_size = 16
train_loader, val_loader, test_loader, class_weights, (X_train, y_train, X_val, y_val, X_test, y_test) = \
    get_data_loaders(dataset_path="APTOS2019", batch_size=batch_size)

total_samples = len(X_train) + len(X_val) + len(X_test)
print(f"\n✓ Dataset Loaded:")
print(f"  Train: {len(X_train):5d} ({len(X_train)/total_samples*100:5.1f}%)")
print(f"  Valid: {len(X_val):5d} ({len(X_val)/total_samples*100:5.1f}%)")
print(f"  Test:  {len(X_test):5d} ({len(X_test)/total_samples*100:5.1f}%)")
print(f"  Total: {total_samples:5d}")
print(f"\n  Batch Size: {batch_size}")
print(f"  Train Batches: {len(train_loader)}")
print(f"  Val Batches: {len(val_loader)}")
print(f"  Test Batches: {len(test_loader)}")


📊 Loading APTOS2019 Dataset...
Total training samples: 3662
Class distribution:
diagnosis
0    1805
1     370
2     999
3     193
4     295
Name: count, dtype: int64

--- Öngörülen Veri Dağılımı ---
Eğitim (Train) : ~2929 örnek (%80.0)
Doğrulama (Val): ~366 örnek (%10.0)
Test (Test)    : ~366 örnek (%10.0)
-------------------------------
Resimler diske yüklenmeye başlıyor, lütfen bekleyin...

Loaded 100/3662 images
Loaded 200/3662 images
Loaded 300/3662 images
Loaded 400/3662 images
Loaded 500/3662 images
Loaded 600/3662 images
Loaded 700/3662 images
Loaded 800/3662 images
Loaded 900/3662 images
Loaded 1000/3662 images
Loaded 1100/3662 images
Loaded 1200/3662 images
Loaded 1300/3662 images
Loaded 1400/3662 images
Loaded 1500/3662 images
Loaded 1600/3662 images
Loaded 1700/3662 images
Loaded 1800/3662 images
Loaded 1900/3662 images
Loaded 2000/3662 images
Loaded 2100/3662 images
Loaded 2200/3662 images
Loaded 2300/3662 images
Loaded 2400/3662 images
Loaded 2500/3662 images
Loaded 2600/

## Training Configuration

The trainer uses:
- **Optimizer**: AdamW (lr=1e-3, weight_decay=1e-4)
- **Scheduler**: Cosine Annealing with Warm Restarts
- **Loss**: Cross Entropy with class weighting
- **Early Stopping**: 15 epochs patience on QWK metric
- **Checkpointing**: Every epoch (auto-resume)

In [10]:
# 5. Define Training Class (Simplified version for notebook)

class HybridModelTrainer:
    """Simplified trainer for notebook execution."""
    
    def __init__(self, model_name: str, config: Dict, device: str = "cuda", 
                 num_epochs: int = 100, batch_size: int = 16, learning_rate: float = 1e-3):
        self.model_name = model_name
        self.config = config
        self.device = device
        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.lr = learning_rate
        
        # Paths
        self.model_dir = results_dir / model_name
        self.model_dir.mkdir(exist_ok=True)
        self.checkpoint_path = self.model_dir / "checkpoint_current.pth"
        self.best_model_path = self.model_dir / "best_model.pth"
        self.metrics_path = self.model_dir / "metrics.json"
        self.history_path = self.model_dir / "training_history.json"
        
        # Model
        self.model = HybridCNNViTBase(num_classes=5, config=config).to(device)
        self.optimizer = optim.AdamW(self.model.parameters(), lr=learning_rate, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(self.optimizer, T_0=10, T_mult=2)
        self.criterion = nn.CrossEntropyLoss()
        
        # State
        self.history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": [], "val_qwk": []}
        self.best_qwk = -np.inf
        self.start_epoch = 0
        
        self._load_checkpoint()
        print(f"✓ Trainer initialized: {model_name}")
    
    def _load_checkpoint(self):
        """Load checkpoint if exists."""
        if self.checkpoint_path.exists():
            checkpoint = torch.load(self.checkpoint_path, map_location=self.device)
            self.model.load_state_dict(checkpoint["model_state"])
            self.optimizer.load_state_dict(checkpoint["optimizer_state"])
            self.history = checkpoint["history"]
            self.best_qwk = checkpoint["best_qwk"]
            self.start_epoch = checkpoint["epoch"] + 1
            print(f"  → Resumed from epoch {self.start_epoch} (Best QWK: {self.best_qwk:.4f})")
    
    def train_epoch(self, train_loader: DataLoader) -> float:
        """Train single epoch."""
        self.model.train()
        total_loss = 0
        
        for images, labels in train_loader:
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(images)
            label_indices = labels.argmax(1) if labels.ndim > 1 else labels
            loss = self.criterion(outputs, label_indices)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()
            
            total_loss += loss.item() * images.size(0)
        
        return total_loss / len(train_loader.dataset)
    
    def validate(self, val_loader: DataLoader) -> Tuple[float, float, float, float]:
        """Validate on validation set."""
        self.model.eval()
        total_loss = 0
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(images)
                label_indices = labels.argmax(1) if labels.ndim > 1 else labels
                loss = self.criterion(outputs, label_indices)
                total_loss += loss.item() * images.size(0)
                preds = outputs.argmax(1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(label_indices.cpu().numpy())
        
        val_loss = total_loss / len(val_loader.dataset)
        accuracy, f1, qwk = compute_metrics(all_labels, all_preds)
        return val_loss, accuracy, f1, qwk
    
    def train(self, train_loader: DataLoader, val_loader: DataLoader):
        """Main training loop."""
        patience, patience_counter = 15, 0
        
        for epoch in range(self.start_epoch, self.num_epochs):
            train_loss = self.train_epoch(train_loader)
            val_loss, val_acc, val_f1, val_qwk = self.validate(val_loader)
            
            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_acc"].append(val_acc)
            self.history["val_f1"].append(val_f1)
            self.history["val_qwk"].append(val_qwk)
            
            # Save checkpoint
            checkpoint = {
                "epoch": epoch,
                "model_state": self.model.state_dict(),
                "optimizer_state": self.optimizer.state_dict(),
                "scheduler_state": self.scheduler.state_dict(),
                "history": self.history,
                "best_qwk": self.best_qwk,
            }
            torch.save(checkpoint, self.checkpoint_path)
            
            # Early stopping
            if val_qwk > self.best_qwk:
                self.best_qwk = val_qwk
                torch.save(self.model.state_dict(), self.best_model_path)
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"  → Early stopping at epoch {epoch+1}")
                    break
            
            self.scheduler.step()
            
            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1:3d}: Loss={train_loss:.4f}→{val_loss:.4f} | Acc={val_acc:.4f} QWK={val_qwk:.4f}")
        
        with open(self.history_path, 'w') as f:
            json.dump(self.history, f)
    
    def evaluate(self, test_loader: DataLoader) -> Dict:
        """Evaluate on test set."""
        self.model.load_state_dict(torch.load(self.best_model_path, map_location=self.device))
        self.model.eval()
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)
                outputs = self.model(images)
                label_indices = labels.argmax(1) if labels.ndim > 1 else labels
                preds = outputs.argmax(1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(label_indices.cpu().numpy())
        
        accuracy, f1, qwk = compute_metrics(all_labels, all_preds)
        cm = confusion_matrix(all_labels, all_preds)
        class_names = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
        report = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
        
        results = {
            "accuracy": float(accuracy),
            "f1_macro": float(f1),
            "qwk": float(qwk),
            "confusion_matrix": cm.tolist(),
            "classification_report": report,
        }
        
        with open(self.metrics_path, 'w') as f:
            json.dump({
                "accuracy": float(accuracy),
                "f1_macro": float(f1),
                "qwk": float(qwk),
                "classification_report": report,
            }, f, indent=2)
        
        return results

print("✓ Trainer class defined")

✓ Trainer class defined


## Training Phase: 10 Models

Starting training of all 10 models with checkpoint support.
Each model trains for up to 100 epochs with early stopping on QWK.

⚠️ **Note**: This section takes significant time (~4-40 hours per model on GPU).
If interrupted, just re-run this cell to continue from checkpoint.

In [11]:
# 6. Train All 10 Models

print("\n" + "="*70)
print("HYBRID CNN+ViT: TRAINING 10 RESEARCH IDEAS")
print("="*70)

all_results = {}
training_summary = []

for idx, model_name in enumerate(list_all_models(), 1):
    print(f"\n{'#'*70}")
    print(f"  MODEL {idx}/10: {model_name}")
    print(f"{'#'*70}")
    
    config = get_model_config(model_name)
    
    try:
        trainer = HybridModelTrainer(
            model_name=model_name,
            config=config,
            device=device,
            num_epochs=100,
            batch_size=16,
            learning_rate=1e-3
        )
        
        print(f"\n  Configuration:")
        print(f"    - Fusion: {config.get('fusion_method', 'N/A')}")
        print(f"    - Uncertainty Refinement: {config.get('use_uncertainty_refinement', False)}")
        print(f"    - Ordinal Head: {config.get('use_ordinal_head', False)}")
        print(f"    - Prototype Memory: {config.get('use_prototype_memory', False)}")
        
        print(f"\n  Training...")
        trainer.train(train_loader, val_loader)
        
        print(f"\n  Evaluating on test set...")
        test_results = trainer.evaluate(test_loader)
        
        all_results[model_name] = test_results
        training_summary.append({
            "Model": model_name.replace("Idea_", ""),
            "Accuracy": test_results["accuracy"],
            "F1": test_results["f1_macro"],
            "QWK": test_results["qwk"],
            "Best_Epoch_QWK": float(trainer.best_qwk),
        })
        
        print(f"\n  ✓ Results:")
        print(f"    - Accuracy: {test_results['accuracy']:.4f}")
        print(f"    - F1-Score: {test_results['f1_macro']:.4f}")
        print(f"    - QWK: {test_results['qwk']:.4f}")
        
    except Exception as e:
        print(f"\n  ✗ Error: {str(e)}")
        all_results[model_name] = {"error": str(e)}

print("\n" + "="*70)
print("TRAINING COMPLETED")
print("="*70)


HYBRID CNN+ViT: TRAINING 10 RESEARCH IDEAS

######################################################################
  MODEL 1/10: Idea_1_ConfidenceGatedFusion
######################################################################
✓ Trainer initialized: Idea_1_ConfidenceGatedFusion

  Configuration:
    - Fusion: gate
    - Uncertainty Refinement: False
    - Ordinal Head: False
    - Prototype Memory: False

  Training...

  ✗ Error: Sizes of tensors must match except in dimension 1. Expected size 16 but got size 196 for tensor number 1 in the list.

######################################################################
  MODEL 2/10: Idea_2_MultiScaleLesionAttention
######################################################################
✓ Trainer initialized: Idea_2_MultiScaleLesionAttention

  Configuration:
    - Fusion: concat
    - Uncertainty Refinement: False
    - Ordinal Head: False
    - Prototype Memory: False

  Training...

  ✗ Error: Sizes of tensors must match except in di

In [7]:
# 7. Create Comparison Summary

comparison_df = pd.DataFrame(training_summary).sort_values("QWK", ascending=False)

print("\n📊 PERFORMANCE RANKING (by QWK):\n")
print(comparison_df.to_string(index=False))

# Save results
comparison_df.to_csv(results_dir / "model_comparison_sorted.csv", index=False)
print(f"\n✓ Saved: {results_dir / 'model_comparison_sorted.csv'}")

# Statistics
print(f"\n📈 STATISTICS:")
print(f"  Average QWK: {comparison_df['QWK'].mean():.4f}")
print(f"  Std Dev QWK: {comparison_df['QWK'].std():.4f}")
print(f"  Max QWK: {comparison_df['QWK'].max():.4f}")
print(f"  Min QWK: {comparison_df['QWK'].min():.4f}")

# Best model
best_idx = comparison_df["QWK"].idxmax()
best_model = comparison_df.loc[best_idx]
print(f"\n🏆 BEST MODEL:")
print(f"  {best_model['Model']}")
print(f"  QWK: {best_model['QWK']:.4f} | Acc: {best_model['Accuracy']:.4f} | F1: {best_model['F1']:.4f}")

KeyError: 'QWK'

In [ ]:
# 8. Visualization: Ranking Comparison

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("10 Models: Performance Rankings", fontsize=14, fontweight='bold')

metrics_to_plot = ["Accuracy", "F1", "QWK"]
colors = plt.cm.viridis(np.linspace(0, 1, len(comparison_df)))

for ax, metric in zip(axes, metrics_to_plot):
    sorted_df = comparison_df.sort_values(metric, ascending=True)
    ax.barh(sorted_df["Model"], sorted_df[metric], color=colors)
    ax.set_xlabel(metric, fontweight='bold')
    ax.set_title(f"{metric} Ranking")
    ax.grid(axis='x', alpha=0.3)
    
    for i, v in enumerate(sorted_df[metric]):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(results_dir / "00_ranking_comparison.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 00_ranking_comparison.png")
plt.show()

In [ ]:
# 9. Load and Plot Training Curves

print("\nPlotting training curves for all models...")

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Training and Validation Loss Curves (All Models)", fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (ax, model_name) in enumerate(zip(axes, list_all_models())):
    history_path = results_dir / model_name / "training_history.json"
    
    if history_path.exists():
        with open(history_path) as f:
            history = json.load(f)
        
        train_loss = history.get("train_loss", [])
        val_loss = history.get("val_loss", [])
        
        ax.plot(train_loss, label="Train", alpha=0.8, linewidth=1.5)
        ax.plot(val_loss, label="Val", alpha=0.8, linewidth=1.5)
        ax.set_title(model_name.replace("Idea_", "Idea "), fontsize=9)
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel("Loss", fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(results_dir / "01_loss_curves_comparison.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 01_loss_curves_comparison.png")
plt.show()

In [ ]:
# 10. Validation Metrics Evolution

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle("Validation QWK and Accuracy Evolution", fontsize=14, fontweight='bold')
axes = axes.flatten()

for idx, (ax, model_name) in enumerate(zip(axes, list_all_models())):
    history_path = results_dir / model_name / "training_history.json"
    
    if history_path.exists():
        with open(history_path) as f:
            history = json.load(f)
        
        val_qwk = history.get("val_qwk", [])
        val_acc = history.get("val_acc", [])
        
        ax.plot(val_qwk, label="QWK", alpha=0.8, linewidth=1.5, marker='o', markersize=3)
        ax.plot(val_acc, label="Accuracy", alpha=0.8, linewidth=1.5, marker='s', markersize=3)
        ax.set_title(model_name.replace("Idea_", "Idea "), fontsize=9)
        ax.set_xlabel("Epoch", fontsize=8)
        ax.set_ylabel("Score", fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7)
        ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(results_dir / "02_validation_metrics_evolution.png", dpi=150, bbox_inches='tight')
print("✓ Saved: 02_validation_metrics_evolution.png")
plt.show()